[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sharukh01621/first-line-to-first-paper/blob/main/colab_drive/solutions/part_07_research_capstone/ch34_error_analysis_fairness_calibration/02_guided_todos_SOLUTION.ipynb)

> **From First Line to First Paper** — *Mastering Python, Statistical Data Science, and Applied Deep Learning*  
> **Author:** Mezanur Rahman Sharukh | **Mode:** `ANNOTATED SOLUTION NOTEBOOK`


# Chapter 34 Guided To-Do Practice Labs

**Book Reference:** Chapter 34, Stage 5 (Guided To-Do Practice)  
Run **Step 0** to activate `sharukh-tutor`, implement all 6 fairness, calibration, error-slicing, threshold-mitigation, and drift-monitoring functions, and pass each **function name (without parentheses!)** to `tutor.check(...)`. Use `tutor.hint("ch34_todo_XX", level=1..3)` whenever you want a progressive nudge.

In [ ]:
# @title Step 0: Click Play (▶) to Activate Interactive Hints & Auto-Checker
!pip install -q "git+https://github.com/sharukh01621/first-line-to-first-paper.git#subdirectory=sharukh_tutor"
import numpy as np
import pandas as pd
from sklearn.metrics import brier_score_loss, roc_auc_score
import sharukh_tutor as tutor

tutor.activate(chapter="ch34", notebook="guided_todos")


---
## To-Do 34.1 (Subgroup Disaggregation): `subgroup_performance_table(y_true, y_prob, groups, threshold=0.5)`

Implement `subgroup_performance_table(y_true, y_prob, groups, threshold=0.5)`:
- For each sorted unique group in `groups`, compute `n` (`int`), `prevalence`, `roc_auc` (`np.nan` if only one class is present in `y_true`), `tpr` (sensitivity), `fpr`, and `brier` (`brier_score_loss`).
- Guard zero denominators for `tpr` and `fpr` cleanly with `0.0`.
- Return a `pd.DataFrame` with exact columns `["group", "n", "prevalence", "roc_auc", "tpr", "fpr", "brier"]` rounded to **4 decimals**.

In [ ]:
def subgroup_performance_table(y_true, y_prob, groups, threshold=0.5):
    """Disaggregate clinical classifier metrics by sorted subgroup."""
    yt = np.asarray(y_true, dtype=int)
    yp = np.asarray(y_prob, dtype=float)
    gr = np.asarray(groups)
    rows = []
    for g in sorted(pd.unique(gr)):
        m = gr == g
        yt_g, yp_g = yt[m], yp[m]
        pred_g = (yp_g >= threshold).astype(int)
        auc = (
            round(float(roc_auc_score(yt_g, yp_g)), 4)
            if len(np.unique(yt_g)) == 2
            else np.nan
        )
        tp = int(np.sum((yt_g == 1) & (pred_g == 1)))
        fn = int(np.sum((yt_g == 1) & (pred_g == 0)))
        fp = int(np.sum((yt_g == 0) & (pred_g == 1)))
        tn = int(np.sum((yt_g == 0) & (pred_g == 0)))
        tpr = tp / (tp + fn) if (tp + fn) > 0 else 0.0
        fpr = fp / (fp + tn) if (fp + tn) > 0 else 0.0
        rows.append({
            "group": g.item() if hasattr(g, "item") else g,
            "n": int(np.sum(m)),
            "prevalence": round(float(np.mean(yt_g == 1)), 4),
            "roc_auc": auc,
            "tpr": round(tpr, 4),
            "fpr": round(fpr, 4),
            "brier": round(float(brier_score_loss(yt_g, yp_g)), 4),
        })
    cols = [
        "group", "n", "prevalence", "roc_auc", "tpr", "fpr", "brier",
    ]
    return pd.DataFrame(rows, columns=cols)


In [ ]:
tutor.check("ch34_todo_01", subgroup_performance_table)


---
## To-Do 34.2 (Fairness Gap Audit): `compute_fairness_gaps(y_true, y_pred, sensitive_attr)`

Implement `compute_fairness_gaps(y_true, y_pred, sensitive_attr)`:
- For each unique group in `sensitive_attr`, compute selection rate (`mean(y_pred == 1)`), TPR, and FPR.
- Compute `max - min` across groups for each rate, and set `equalized_odds_diff = max(equal_opportunity_diff, max_fpr_diff)`.
- Return `{"demographic_parity_diff": round4, "equal_opportunity_diff": round4, "max_fpr_diff": round4, "equalized_odds_diff": round4}`.

In [ ]:
def compute_fairness_gaps(y_true, y_pred, sensitive_attr):
    """Compute max-min DP, Equal Opportunity, FPR, and EqOdds gaps."""
    yt = np.asarray(y_true, dtype=int)
    yp = np.asarray(y_pred, dtype=int)
    gr = np.asarray(sensitive_attr)
    if not (len(yt) == len(yp) == len(gr)):
        raise ValueError(
            "y_true, y_pred, and sensitive_attr must have equal length"
        )
    sels, tprs, fprs = [], [], []
    for g in sorted(pd.unique(gr)):
        m = gr == g
        yt_g, yp_g = yt[m], yp[m]
        sels.append(float(np.mean(yp_g == 1)))
        tp = int(np.sum((yt_g == 1) & (yp_g == 1)))
        fn = int(np.sum((yt_g == 1) & (yp_g == 0)))
        fp = int(np.sum((yt_g == 0) & (yp_g == 1)))
        tn = int(np.sum((yt_g == 0) & (yp_g == 0)))
        tprs.append(tp / (tp + fn) if (tp + fn) > 0 else 0.0)
        fprs.append(fp / (fp + tn) if (fp + tn) > 0 else 0.0)
    eo = max(tprs) - min(tprs)
    fpr_d = max(fprs) - min(fprs)
    return {
        "demographic_parity_diff": round(max(sels) - min(sels), 4),
        "equal_opportunity_diff": round(eo, 4),
        "max_fpr_diff": round(fpr_d, 4),
        "equalized_odds_diff": round(max(eo, fpr_d), 4),
    }


In [ ]:
tutor.check("ch34_todo_02", compute_fairness_gaps)


---
## To-Do 34.3 (Subgroup Calibration): `subgroup_calibration_audit(y_true, y_prob, groups, n_bins=5)`

Implement `subgroup_calibration_audit(y_true, y_prob, groups, n_bins=5)`:
- For each sorted unique group, compute equal-width Expected Calibration Error (ECE) on `[0, 1]` (making the last bin closed `[lo, 1.0]` so `p == 1.0` is included) and Brier score.
- Return `{"per_group": DataFrame(["group", "n", "ece", "brier"]), "worst_ece_group": str, "max_ece_gap": round4}`.

In [ ]:
def subgroup_calibration_audit(y_true, y_prob, groups, n_bins=5):
    """Audit equal-width ECE and Brier score across sorted subgroups."""
    yt = np.asarray(y_true, dtype=float)
    yp = np.asarray(y_prob, dtype=float)
    gr = np.asarray(groups)
    edges = np.linspace(0.0, 1.0, n_bins + 1)
    rows, raw_eces = [], {}
    for g in sorted(pd.unique(gr)):
        m = gr == g
        yt_g, yp_g = yt[m], yp[m]
        n_g = int(np.sum(m))
        ece = 0.0
        for i in range(n_bins):
            lo, hi = edges[i], edges[i + 1]
            if i < n_bins - 1:
                bm = (yp_g >= lo) & (yp_g < hi)
            else:
                bm = (yp_g >= lo) & (yp_g <= hi)
            cnt = int(np.sum(bm))
            if cnt > 0:
                gap = abs(float(yt_g[bm].mean()) - float(yp_g[bm].mean()))
                ece += (cnt / n_g) * gap
        g_val = g.item() if hasattr(g, "item") else g
        raw_eces[str(g_val)] = ece
        rows.append({
            "group": g_val,
            "n": n_g,
            "ece": round(ece, 4),
            "brier": round(float(brier_score_loss(yt_g, yp_g)), 4),
        })
    worst = max(raw_eces, key=raw_eces.get)
    max_gap = round(max(raw_eces.values()) - min(raw_eces.values()), 4)
    return {
        "per_group": pd.DataFrame(
            rows, columns=["group", "n", "ece", "brier"]
        ),
        "worst_ece_group": worst,
        "max_ece_gap": max_gap,
    }


In [ ]:
tutor.check("ch34_todo_03", subgroup_calibration_audit)


---
## To-Do 34.4 (Covariate Shift PSI): `population_stability_index(expected, actual, n_bins=10, eps=1e-6)`

Implement `population_stability_index(expected, actual, n_bins=10, eps=1e-6)`:
- Build quantile bin edges from `expected` via `np.unique(np.quantile(expected, np.linspace(0, 1, n_bins + 1)))` and replace outer edges with `-np.inf` and `+np.inf` so every value in `actual` bins cleanly.
- Compute bin shares for `expected` and `actual`, clip to `>= eps`, re-normalize each vector to sum to `1.0`, and sum `(p_act - p_exp) * np.log(p_act / p_exp)`.
- Return `{"psi": round4, "drift_status": str}` (`"Stable"` `< 0.10`, `"Moderate Drift"` `< 0.25`, `"Severe Drift"` `>= 0.25`).

In [ ]:
def population_stability_index(expected, actual, n_bins=10, eps=1e-6):
    """Compute PSI and drift severity status from quantile bins."""
    exp = np.asarray(expected, dtype=float)
    act = np.asarray(actual, dtype=float)
    if len(exp) == 0 or len(act) == 0:
        raise ValueError("expected and actual arrays must be non-empty")
    edges = np.unique(np.quantile(exp, np.linspace(0.0, 1.0, n_bins + 1)))
    if len(edges) < 2:
        edges = np.array([-np.inf, np.inf])
    else:
        edges = edges.copy()
        edges[0] = -np.inf
        edges[-1] = np.inf
    exp_cnt, _ = np.histogram(exp, bins=edges)
    act_cnt, _ = np.histogram(act, bins=edges)
    p_exp = np.clip(exp_cnt / exp_cnt.sum(), eps, None)
    p_exp = p_exp / p_exp.sum()
    p_act = np.clip(act_cnt / act_cnt.sum(), eps, None)
    p_act = p_act / p_act.sum()
    psi = round(float(np.sum((p_act - p_exp) * np.log(p_act / p_exp))), 4)
    if psi < 0.10:
        status = "Stable"
    elif psi < 0.25:
        status = "Moderate Drift"
    else:
        status = "Severe Drift"
    return {"psi": psi, "drift_status": status}


In [ ]:
tutor.check("ch34_todo_04", population_stability_index)


---
## To-Do 34.5 (Error Slicing): `slice_worst_errors(df, y_true, y_prob, top_k=5)`

Implement `slice_worst_errors(df, y_true, y_prob, top_k=5)`:
- Work on a non-mutating copy of `df` and attach `y_true` and `y_prob`.
- `worst_false_negatives`: rows with `y_true == 1` sorted by `y_prob` ascending (`head(top_k)`), index reset.
- `worst_false_positives`: rows with `y_true == 0` sorted by `y_prob` descending (`head(top_k)`), index reset.
- Return `{"worst_false_negatives": DataFrame, "worst_false_positives": DataFrame}`.

In [ ]:
def slice_worst_errors(df, y_true, y_prob, top_k=5):
    """Return top-k most confident false negatives and false positives."""
    work = df.copy()
    work["y_true"] = np.asarray(y_true, dtype=int)
    work["y_prob"] = np.asarray(y_prob, dtype=float)
    fn_df = (
        work[work["y_true"] == 1]
        .sort_values("y_prob", ascending=True)
        .head(top_k)
        .reset_index(drop=True)
    )
    fp_df = (
        work[work["y_true"] == 0]
        .sort_values("y_prob", ascending=False)
        .head(top_k)
        .reset_index(drop=True)
    )
    return {
        "worst_false_negatives": fn_df,
        "worst_false_positives": fp_df,
    }


In [ ]:
tutor.check("ch34_todo_05", slice_worst_errors)


---
## To-Do 34.6 (Group Threshold Calibration): `calibrate_group_thresholds_for_tpr(y_true, y_prob, groups, target_tpr=0.75)`

Implement `calibrate_group_thresholds_for_tpr(y_true, y_prob, groups, target_tpr=0.75)`:
- Validate `0 < target_tpr <= 1` (raise `ValueError` otherwise).
- For each unique group, find the **highest** threshold among sorted unique probabilities in that group achieving `tpr >= target_tpr` (fallback `0.0` if unreachable or if the group has no positive labels).
- Return a dictionary mapping `group -> round4_float`.

In [ ]:
def calibrate_group_thresholds_for_tpr(
    y_true, y_prob, groups, target_tpr=0.75
):
    """Pick highest per-group cutoff achieving TPR >= target_tpr."""
    if not (0.0 < target_tpr <= 1.0):
        raise ValueError("target_tpr must lie in (0, 1]")
    yt = np.asarray(y_true, dtype=int)
    yp = np.asarray(y_prob, dtype=float)
    gr = np.asarray(groups)
    out = {}
    for g in sorted(pd.unique(gr)):
        m = gr == g
        yt_g, yp_g = yt[m], yp[m]
        n_pos = int(np.sum(yt_g == 1))
        best_t = 0.0
        if n_pos > 0:
            for t in np.sort(np.unique(yp_g))[::-1]:
                if np.sum((yt_g == 1) & (yp_g >= t)) / n_pos >= target_tpr:
                    best_t = float(t)
                    break
        out[g.item() if hasattr(g, "item") else g] = round(best_t, 4)
    return out


In [ ]:
tutor.check("ch34_todo_06", calibrate_group_thresholds_for_tpr)


---
## Annotated Debugging Post-Mortem (Solution Notebook Only)

- **To-Do 34.1 (Single-Class Subgroup AUC Crash):** In small clinical strata (such as the `Middle` wealth quintile on test splits), a subgroup can contain zero positive cases. Calling `roc_auc_score` directly throws an error or warning; always check `len(np.unique(yt_g)) == 2` and emit `np.nan` otherwise.
- **To-Do 34.4 (Out-of-Range Values in PSI):** If you keep finite quantile endpoints `[min(expected), max(expected)]`, extreme shifted values in `actual` (e.g., severe famine anemia below historical minimum Hb) fall outside `np.histogram` bins and vanish! Replacing outer edges with `-np.inf` and `+np.inf` captures every shifted patient cleanly.
- **To-Do 34.6 (Highest vs. Lowest Qualifying Threshold):** Choosing the *lowest* threshold where `TPR >= target_tpr` trivially predicts `1` for everyone (`FPR = 1.0`). Scanning unique probabilities in descending order and stopping at the *highest* qualifying cutoff hits `target_tpr` with minimal false alarms.

In [ ]:
tutor.summary()
